# Complete FastHTML Application Example

This notebook demonstrates a complete application with FastHTML's SSE support, HTMX integration, multiple endpoints, error handling, and job status tracking.

In [1]:
from fasthtml.common import *
import uuid, time, json

from cjm_tqdm_capture.progress_monitor import ProgressMonitor
from cjm_tqdm_capture.job_runner import JobRunner
from cjm_tqdm_capture.streaming import sse_stream
from cjm_tqdm_capture.progress_info import serialize_all_jobs

In [2]:
# For Jupyter display
from fasthtml.jupyter import JupyUvi, HTMX
from cjm_fasthtml_daisyui.core.testing import create_test_app, create_test_page, start_test_server
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from IPython.display import display

# Import DaisyUI factories
from cjm_fasthtml_daisyui.components.actions.button import btn, btn_colors, btn_sizes
from cjm_fasthtml_daisyui.components.feedback.progress import progress, progress_colors
from cjm_fasthtml_daisyui.components.data_display.card import card
from cjm_fasthtml_daisyui.components.data_input.text_input import text_input
from cjm_fasthtml_daisyui.components.data_input.select import select
from cjm_fasthtml_daisyui.components.data_display.table import table, table_modifiers
from cjm_fasthtml_daisyui.components.data_input.label import label
from cjm_fasthtml_daisyui.utilities.semantic_colors import bg_dui

# Import Tailwind factories
from cjm_fasthtml_tailwind.utilities.spacing import p, m, space
from cjm_fasthtml_tailwind.utilities.typography import font_weight, font_size
from cjm_fasthtml_tailwind.utilities.sizing import w, max_w, container
from cjm_fasthtml_tailwind.utilities.layout import overflow
from cjm_fasthtml_tailwind.utilities.borders import rounded
from cjm_fasthtml_tailwind.core.base import combine_classes

In [3]:
# Create app
from cjm_fasthtml_daisyui.core.themes import DaisyUITheme
from cjm_fasthtml_daisyui.core.testing import create_test_app

app, rt = create_test_app(theme=DaisyUITheme.DARK)

# Initialize with history for debugging
monitor = ProgressMonitor(keep_history=True, history_limit=100)
runner = JobRunner(monitor)

# Store job results - renamed to avoid conflict with endpoint function
job_results_store = {}

In [4]:
# Define a complex task with parameters and multiple stages
def complex_task(task_name, iterations, speed="normal"):
    from tqdm import tqdm
    import time
    import random
    
    speeds = {"fast": 0.001, "normal": 0.01, "slow": 0.05}
    delay = speeds.get(speed, 0.01)
    
    results = {"task": task_name, "stages": []}
    
    # Stage 1: Initialization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Initialize"):
        time.sleep(delay)
    results["stages"].append("Initialization complete")
    
    # Stage 2: Processing
    processed_data = []
    for i in tqdm(range(iterations), desc=f"{task_name}: Process"):
        time.sleep(delay)
        processed_data.append(i * random.random())
    results["stages"].append(f"Processed {len(processed_data)} items")
    
    # Stage 3: Validation
    for i in tqdm(range(iterations // 2), desc=f"{task_name}: Validate"):
        time.sleep(delay)
    results["stages"].append("Validation complete")
    
    # Stage 4: Finalization
    for i in tqdm(range(iterations // 4), desc=f"{task_name}: Finalize"):
        time.sleep(delay)
    results["stages"].append("Finalization complete")
    
    results["summary"] = f"Task {task_name} completed successfully with {iterations} iterations"
    return results

In [5]:
# Enhanced UI with HTMX polling approach
from cjm_fasthtml_daisyui.core.testing import create_test_page

@rt("/")
def index():
    return create_test_page(
        "Complete Progress Application",
        Div(
            H1("Task Manager with Progress Tracking", cls=combine_classes(font_size._3xl, font_weight.bold, m.b(6))),
            
            # Task configuration form
            Form(
                H2("Configure Task", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Label("Task Name:", cls=str(label), fr="task-name"),
                    Input(id="task-name", name="task_name", type="text", value="MyTask", 
                          cls=combine_classes(text_input, w.full, max_w.xs)),
                    cls=str(m.b(4))
                ),
                Div(
                    Label("Iterations:", cls=str(label), fr="iterations"),
                    Input(id="iterations", name="iterations", type="number", value="100", 
                          min="10", max="500", cls=combine_classes(text_input, w.full, max_w.xs)),
                    cls=str(m.b(4))
                ),
                Div(
                    Label("Speed:", cls=str(label), fr="speed"),
                    Select(
                        Option("Fast", value="fast"),
                        Option("Normal", value="normal", selected=True),
                        Option("Slow", value="slow"),
                        id="speed",
                        name="speed",
                        cls=combine_classes(select, w.full, max_w.xs)
                    ),
                    cls=str(m.b(4))
                ),
                Button("Start Task", type="submit", id="start-btn", 
                       cls=combine_classes(btn, btn_colors.primary)),
                Button("Clear Completed", type="button", 
                       hx_post="/clear_jobs",
                       hx_target="#jobs-container",
                       hx_swap="innerHTML",
                       cls=combine_classes(btn, btn_colors.secondary, m.l(2))),
                hx_post="/create_job",
                hx_target="#progress-container",
                hx_swap="innerHTML",
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Progress display container
            Div(
                H2("Current Progress", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    P("No active job", cls=str(font_size.sm)),
                    id="progress-container"
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6), m.b(6))
            ),
            
            # Active jobs list
            Div(
                H2("Active Jobs", cls=combine_classes(font_size.xl, font_weight.semibold, m.b(4))),
                Div(
                    Div("Loading jobs...", id="jobs-list"),
                    hx_get="/jobs_table",
                    hx_trigger="load, every 2s",
                    hx_swap="innerHTML",
                    id="jobs-container",
                    cls=str(overflow.x.auto)
                ),
                cls=combine_classes(card, bg_dui.base_200, p(6))
            ),
            
            cls=combine_classes(container, m.x.auto, p(8))
        )
    )

In [6]:
# API endpoints using FastHTML conventions with HTMX polling
@rt("/create_job")
def create_job(task_name: str, iterations: int, speed: str):
    job_id = str(uuid.uuid4())
    
    # Wrapper to store results
    def task_wrapper():
        result = complex_task(task_name, iterations, speed)
        job_results_store[job_id] = result
        return result
    
    # Adjust throttling based on speed
    patch_config = {
        'fast': {"min_delta_pct": 10, "min_update_interval": 0.01, "emit_initial": True},
        'normal': {"min_delta_pct": 5, "min_update_interval": 0.05, "emit_initial": True},
        'slow': {"min_delta_pct": 2, "min_update_interval": 0.2, "emit_initial": True}
    }
    
    runner.start(
        job_id,
        task_wrapper,
        patch_kwargs=patch_config.get(speed, patch_config['normal'])
    )
    
    # Return two separate sections: progress (polls frequently) and results (polls less frequently)
    return Div(
        H3(f"Task: {task_name}", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(2))),
        P(f"Job ID: {job_id[:8]}...", cls=combine_classes(font_size.xs, m.b(4))),
        
        # Progress section - polls every 100ms until complete
        Div(
            # Initial state
            P("Overall: 0%", cls=str(font_weight.bold)),
            Progress(value="0", max="100", 
                    cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))),
            P("Starting...", cls=str(font_size.sm)),
            
            # HTMX polling for progress
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load, every 100ms",
            hx_swap="innerHTML",
            id=f"progress-{job_id}"
        ),
        
        # Results section - uses outerHTML swap to replace entire polling div
        Div(
            # Initially empty, will show results when complete
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load, every 500ms",
            hx_swap="outerHTML",  # IMPORTANT: Replace the entire div, not just innerHTML
            id=f"results-{job_id}"
        )
    )

@rt("/job_progress")
def job_progress(job_id: str):
    """Returns current progress for a job - focuses only on progress bars"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        # Job not found or not started yet
        return Div(
            P("Waiting for job to start...", cls=str(font_weight.bold)),
            Progress(value="0", max="100", 
                    cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))),
            # Keep polling
            hx_get=f"/job_progress?job_id={job_id}",
            hx_trigger="load delay:100ms",
            hx_swap="outerHTML"
        )
    
    # Build progress bars dynamically based on actual tqdm bars
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                  cls=combine_classes(font_size.sm, font_weight.semibold)),
                Progress(
                    value=str(bar.progress), 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(2))
            )
        )
    
    # Check if completed - stop polling for progress
    if snapshot['completed']:
        # Return final progress state without polling
        return Div(
            P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
            Progress(
                value=str(snapshot['overall_progress']), 
                max="100", 
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
            *bars_html,
            P("Task completed!", cls=combine_classes(font_size.sm, font_weight.semibold, m.t(2)))
            # NO polling trigger - progress updates stop
        )
    
    # Return current progress with continued polling
    return Div(
        P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
        Progress(
            value=str(snapshot['overall_progress']), 
            max="100", 
            cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
        ),
        *bars_html,
        P(f"Running... ({len(snapshot['bars'])} active bars)", cls=str(font_size.sm)),
        # Continue polling
        hx_get=f"/job_progress?job_id={job_id}",
        hx_trigger="load delay:100ms",
        hx_swap="outerHTML"
    )

@rt("/job_results_section")
def job_results_section(job_id: str):
    """Returns the results section - replaces entire polling div when complete"""
    snapshot = monitor.snapshot(job_id)
    
    # If job not complete or no results yet, keep polling
    if not snapshot or not snapshot['completed']:
        return Div(
            # Empty while running, keep checking
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    
    # Job is complete - check for results
    if job_id not in job_results_store:
        # Still waiting for results to be stored
        return Div(
            P("Processing results...", cls=str(font_size.sm)),
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    
    # Results are available - return static div that REPLACES the polling div
    # This div has the same ID but NO HTMX attributes
    return Div(
        Details(
            Summary("View Results", cls=combine_classes(font_weight.semibold, m.t(4))),
            Pre(
                json.dumps(job_results_store[job_id], indent=2),
                cls=combine_classes(bg_dui.base_300, p(4), rounded(), font_size.xs, m.t(2))
            ),
            open=False
        ),
        id=f"results-{job_id}"  # Same ID, but no HTMX attributes = no more polling
    )

In [7]:
# Jobs table endpoint for HTMX polling
import json  # Add import for JSON serialization

@rt("/jobs_table")
def jobs_table():
    jobs = monitor.all()
    serialized = serialize_all_jobs(jobs)
    
    if not serialized:
        return Div("No active jobs", id="jobs-list")
    
    # Create table with jobs
    rows = []
    for job_id, job_data in serialized.items():
        rows.append(
            Tr(
                Td(job_id[:8] + "..."),
                Td(f"{job_data['overall_progress']:.1f}%"),
                Td("Complete" if job_data['completed'] else "Running"),
                Td(
                    Button("View", 
                           hx_get=f"/view_job?job_id={job_id}",
                           hx_target="#progress-container",
                           hx_swap="innerHTML transition:true",  # Add smooth transition
                           cls=combine_classes(btn, btn_sizes.xs))
                )
            )
        )
    
    return Table(
        Thead(
            Tr(
                Th("Job ID"),
                Th("Progress"),
                Th("Status"),
                Th("Action")
            )
        ),
        Tbody(*rows),
        cls=combine_classes(table, table_modifiers.zebra, w.full),
        id="jobs-list"
    )

@rt("/view_job")
def view_job(job_id: str):
    """View a specific job's progress"""
    snapshot = monitor.snapshot(job_id)
    
    if not snapshot:
        return P("Job not found", cls=combine_classes(font_size.sm))
    
    # Pre-populate with actual current progress to avoid empty state
    bars_html = []
    for bar_id, bar in snapshot['bars'].items():
        bars_html.append(
            Div(
                P(f"{bar.description}: {bar.progress:.1f}% ({bar.current}/{bar.total})", 
                  cls=combine_classes(font_size.sm, font_weight.semibold)),
                Progress(
                    value=str(bar.progress), 
                    max="100", 
                    cls=combine_classes(progress, progress_colors.accent, w.full)
                ),
                cls=str(m.b(2))
            )
        )
    
    # Pre-render results section based on current state
    if snapshot['completed'] and job_id in job_results_store:
        # Job is complete and results are available
        results_section = Div(
            Details(
                Summary("View Results", cls=combine_classes(font_weight.semibold, m.t(4))),
                Pre(
                    json.dumps(job_results_store[job_id], indent=2),
                    cls=combine_classes(bg_dui.base_300, p(4), rounded(), font_size.xs, m.t(2))
                ),
                open=False
            ),
            id=f"results-{job_id}"  # Static div, no polling
        )
    elif snapshot['completed']:
        # Job is complete but results not yet available
        results_section = Div(
            P("Processing results...", cls=str(font_size.sm)),
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load delay:500ms",
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    else:
        # Job is still running - empty div that will poll for results
        results_section = Div(
            hx_get=f"/job_results_section?job_id={job_id}",
            hx_trigger="load delay:500ms, every 500ms",
            hx_swap="outerHTML",
            id=f"results-{job_id}"
        )
    
    # Return everything pre-rendered
    return Div(
        H3(f"Viewing Job", cls=combine_classes(font_size.lg, font_weight.semibold, m.b(2))),
        P(f"Job ID: {job_id[:8]}...", cls=combine_classes(font_size.xs, m.b(4))),
        
        # Progress section - start with actual current state
        Div(
            # Show actual current progress immediately
            P(f"Overall: {snapshot['overall_progress']:.1f}%", cls=str(font_weight.bold)),
            Progress(
                value=str(snapshot['overall_progress']), 
                max="100", 
                cls=combine_classes(progress, progress_colors.primary, w.full, m.b(4))
            ),
            *bars_html,
            P("Task completed!" if snapshot['completed'] else f"Running... ({len(snapshot['bars'])} active bars)", 
              cls=str(font_size.sm)),
            
            # HTMX polling for updates (only if not completed)
            hx_get=f"/job_progress?job_id={job_id}" if not snapshot['completed'] else None,
            hx_trigger="load delay:100ms, every 100ms" if not snapshot['completed'] else None,
            hx_swap="innerHTML" if not snapshot['completed'] else None,
            id=f"progress-{job_id}"
        ),
        
        # Results section - pre-rendered based on state
        results_section
    )

In [8]:
# Get specific job status (for API usage)
@rt("/job_status")
def job_status(job_id: str):
    snapshot = monitor.snapshot(job_id)
    if not snapshot:
        return JSONResponse({"error": "Job not found"}, status_code=404)
    
    return JSONResponse({
        "job_id": job_id,
        "progress": snapshot["overall_progress"],
        "completed": snapshot["completed"],
        "bars": {
            k: {
                "description": v.description,
                "progress": v.progress,
                "current": v.current,
                "total": v.total
            }
            for k, v in snapshot["bars"].items()
        }
    })

In [9]:
# Get job result endpoint (for direct API usage)
@rt("/job_result")  
def job_result(job_id: str):
    """API endpoint to get job results as JSON"""
    if job_id in job_results_store:
        return JSONResponse(job_results_store[job_id])
    return JSONResponse({"error": "No results yet"}, status_code=404)

In [10]:
# SSE stream endpoint using FastHTML's EventStream
@rt("/stream")
def stream(job_id: str):
    return EventStream(
        sse_stream(
            monitor,
            job_id,
            interval=0.1,
            heartbeat=10.0,
            wait_for_start=True,
            start_timeout=30.0
        )
    )

In [11]:
# Clear completed jobs endpoint
@rt("/clear_jobs")
def clear_jobs():
    monitor.clear_completed(older_than_seconds=0)
    # Also clear results for completed jobs
    for job_id in list(job_results_store.keys()):
        if not monitor.snapshot(job_id):
            del job_results_store[job_id]
    
    # Return updated jobs table
    return jobs_table()

In [12]:
# Start server for Jupyter
from cjm_fasthtml_daisyui.core.testing import start_test_server
from fasthtml.jupyter import HTMX
from IPython.display import display

server = start_test_server(app)
display(HTMX())

In [13]:
# Stop server when done
server.stop()